Extraccion y filtrado de parches a partir de imagenes WSI

In [ ]:
#Librerias
import openslide #para leer imagenes en WSI
import numpy as np #para operaciones matematicas y arreglos numericos
import pandas as pd #lectura y organizacion de metadatos
from pathlib import Path #manejo de rutas y carpetas
from PIL import Image #convecion, guardado y manipulacion de imagen
from skimage.color import rgb2hsv, rgb2gray #para transformar imagenes en RGB a otros espacios de color
from skimage.filters import threshold_otsu #para calcular el umbral (separacion de fondo y tejido)
from skimage.morphology import remove_small_objects, remove_small_holes, binary_closing, disk # para operaciones morfologicas (para mejorar la mascara)
from scipy.ndimage import binary_fill_holes #rellenar espacios vacios
from skimage.measure import label, regionprops #para identificar y analizar regiones conectadas

#Define carpetas
BASE_D = Path(r"D:\ProyectoPDAC") # carpeta base del proyecto
SVS_DIR = BASE_D / "Histologia_SVS" / "PDA" #ruta base de imagenes crudas, formato svs
PATCH_DIR = BASE_D / "Parches_PDAC_python" #ruta de parches salida
CSV_LABELS = Path(r"C:\Users\Usuario\Python\Tesis\TCIA CPTAC Pathology Portal.csv") # ruta del archivo maestro (info de slides, pacientes, muestra etc)
MASK_DIR = BASE_D / "Masks_PDAC_python" # Ruta de mascaras generadas. salida

#Crea las carpetas
MASK_DIR.mkdir(parents=True, exist_ok=True)
PATCH_DIR.mkdir(parents=True, exist_ok=True)

#Parametros 
PATCH_SIZE = 256 #tamaño del parche extraido de la WSI original
READ_LEVEL = 0 #nivel de lectura (original, maxima resolucion)
THUMB_MAX_SIZE = 2048 #tamaño maximo de la miniatura (empleado para mascara global)

MIN_TISSUE_RATIO = 0.70 #porcentaje minimo de tejido por parche
WHITE_PIXEL_THR = 220 #umbral de fondo flanco
MAX_WHITE_RATIO = 0.85 #Porcentaje maximo permitido de fondo blanco
MIN_MEAN_SATURATION = 0.02 #saturacion media minima para conservar el parche
MAX_DARK_RATIO = 0.15 #maximo porcentaje de regioens oscura o artefactos

#Parametro para filtros adicionales
MIN_GRAY_STD = 14.0 # desviacion estandar minima en escala de grices (empleado para identificar regioens muy homogeneas y poca variabilidad visual)
MIN_SAT_STD = 0.035 # desviacion estandar minima de saturacion (para descartar regioens con poca variacion de color)
MIN_ENTROPY = 3.0 # entropia minima del parche (para identificar la cantidad de informacion y complejidad de la imagen)
MIN_GRAD_MEAN = 9.0 #gradiente medio minimo (para evaluar la presencia de bordes y estructuras dentro del parche)

#Funciones

#Funcion: THUMBNAIL RGB
def get_thumbnail_rgb(slide, max_size=2048):
    w, h = slide.dimensions #obtenemos las dimenciones originales de la WSI
    scale = min(max_size / w, max_size / h) #calculamos el factor de reduccion manteniendo la proporcion de la imagen
    thumb_size = (max(1, int(w * scale)), max(1, int(h * scale))) #definimos el tamaño final de la miniatura
    thumb = slide.get_thumbnail(thumb_size).convert("RGB") #generamos la miniatura
    return np.array(thumb)# convertimos la imagen a un arreglo matricial NumPy

#Funcion: Detectar regiones que tocan con bordes
def touches_border(region_mask):
    return (
        np.any(region_mask[0, :]) or   #verificamos pixeles activos en los bordes de la mascara
        np.any(region_mask[-1, :]) or
        np.any(region_mask[:, 0]) or
        np.any(region_mask[:, -1])     # Se revisan los 4 bordes (superior, inferior, izquierdo y derecho)
    )

#Funcion: Mascara de tejido
def create_tissue_mask(thumb_rgb):
    thumb_float = thumb_rgb.astype(np.float32) / 255.0 #transformamos a float la imagen RGB
    hsv = rgb2hsv(thumb_float) #Conversion de espacios de color
    gray = rgb2gray(thumb_float)

    s = hsv[:, :, 1] #Canales de saturacion y brillo
    v = hsv[:, :, 2]

    # 1. identifiacion de regiones de tejido
    not_white = gray < 0.95 #filtramos regiones muy blancas
    not_black = v > 0.15    #filtramos regiones muy oscuras
    colored = s > 0.04      #Filtramos regiones con color

    tissue_mask = not_white & not_black & colored #Combinamos las condiciones para crear una mascara principal

    # 2. Alternativa si la máscara sale demasiado pobre
        if tissue_mask.sum() < 500: #si la mascara tiene muy pocos pixeles se aplica el metodo de umbralización automática de Otsu
        thr = threshold_otsu(gray)
        tissue_mask = (gray < thr) & not_black

    # 3. Limpieza morfológica de la mascara
    tissue_mask = binary_closing(tissue_mask, disk(3))                  #une regiones cercanas
    tissue_mask = binary_fill_holes(tissue_mask)                        #rellena todos lo huecos internos
    tissue_mask = remove_small_objects(tissue_mask, min_size=200)       #elimina regiones conectadas muy pequeñas
    tissue_mask = remove_small_holes(tissue_mask, area_threshold=200)   #rellena solo los pequeños espacios vacios

    # 4. Etiquetar componentes
    labeled = label(tissue_mask)       #label busca pixeles activos en la mascara (blancos)
    props = regionprops(labeled)       

    if len(props) == 0:                                       #si no encuentra el tamaño de len es 0, por lo cual sabemos que no se identifico tejido
        return np.zeros_like(tissue_mask, dtype=np.uint8)     #evitamos analizar obtetos que no hay

    # 5. Seleccionamos solo la componente principal de la mascara de tejido
    h_mask, w_mask = tissue_mask.shape  #obtenemos las dimenciones
    center_x = w_mask / 2               #calculamos el centro de la imagen
    center_y = h_mask / 2               
    max_dist = np.sqrt(center_x**2 + center_y**2)  #Distancia maxima posible desde el centro

    candidates = []

    for region in props:              #evaluamos las componetes detectadas
        if region.area < 300:         #filtramos regiones muy pequeñas
            continue

        cy, cx = region.centroid      #obtenemos el centroide de la region
        dist = np.sqrt((cx - center_x) ** 2 + (cy - center_y) ** 2)   #calculamos la distancia al centroide
        dist_norm = dist / max_dist

        score = region.area * (1 - dist_norm)  #calculamos un puntaje basado en el tamaño y la centralidad
        candidates.append((score, region.label))

    if len(candidates) == 0:                 # si no hay candidatos devuelve una mascara vacia
        return np.zeros_like(tissue_mask, dtype=np.uint8)

    best_label = max(candidates, key=lambda x: x[0])[1]  #selecciona la region con mejor porcentaje
    final_mask = (labeled == best_label) #crea la mascara final con la componente principal seleccionada

    return final_mask.astype(np.uint8)

#Funcion: Calcular el porcentaje de tejido en parches
def get_patch_tissue_ratio(mask, x0, y0, patch_size, full_w, full_h):
    mask_h, mask_w = mask.shape  #dimenciones de la mascara de tejido

    scale_x = mask_w / full_w  #calculamos factores de escala entre WSI y la mascara generada a patir de la miniatura
    scale_y = mask_h / full_h

    mx0 = int(x0 * scale_x) #transformacion de coordenadas del parche original al espacio de coordenadas de la mascara
    my0 = int(y0 * scale_y)
    mx1 = int((x0 + patch_size) * scale_x)
    my1 = int((y0 + patch_size) * scale_y)

    mx1 = min(mx1, mask_w)  #limitamos coordenadas para evitar salir de los vordes de la mascara
    my1 = min(my1, mask_h)

    if mx1 <= mx0 or my1 <= my0:    #verificacion de region valida
        return 0.0

    mask_patch = mask[my0:my1, mx0:mx1]  #extraemos la region correspondiente del parche dentro de la mascara

    if mask_patch.size == 0:  #Verificacion de contenido valido
        return 0.0

    tissue_ratio = np.mean(mask_patch > 0)  #calculamos el pocentaje de pixeles pertenecientes al tejido
    return tissue_ratio

#Funcion: Filtro de calidad de parche    
def patch_quality_filters(patch_pil): 
    patch_rgb = np.array(patch_pil)  #convertimos la imagen a arreglo NumPy
    patch_float = patch_rgb.astype(np.float32) / 255.0 #Normalizamos velores RGB entre 0 y 1 para trabajar con HSV

    gray = np.mean(patch_rgb, axis=2) #pasamos la imagen a escala de grices
    hsv = rgb2hsv(patch_float) #usamos la conversion a HSV para evaluar la saturación/color

    s = hsv[:, :, 1] #Canal de saturación

    # 1. Filtros básicos de calidad
    white_ratio = np.mean(gray > WHITE_PIXEL_THR) #proporcion de pixeles muy claros
    mean_saturation = np.mean(s)                  #saturacion media del parche  
    dark_ratio = np.mean(gray < 30)               #proporcion de pixeles muy oscuros

    if white_ratio > MAX_WHITE_RATIO:    #Condiciones de filtrado
        return False

    if mean_saturation < MIN_MEAN_SATURATION:
        return False

    if dark_ratio > MAX_DARK_RATIO:
        return False

    # 2. Variabilidad global o riqueza de texturas
    gray_std = np.std(gray) #calculamos la variacion de intensidad en escala de grises
    sat_std = np.std(s)     #tambien  la variacion de saturacion/color

    # 3. Entropía en escala de grises
    hist, _ = np.histogram(gray, bins=32, range=(0, 255), density=True) #usamos histograma para estimar la complejidad visual
    hist = hist[hist > 0] #evitamos 0 para calculos logaritmicos
    entropy = -np.sum(hist * np.log2(hist))  #calculamos entropia (usada para medir la dispercion de intensidades del parche) 

    # 4. Gradiente medio
    gx = np.diff(gray, axis=1) #calculo de intensidades entre pixeles vecinos (horizontales y verticales)
    gy = np.diff(gray, axis=0)
    grad_mean = np.mean(np.abs(gx)) + np.mean(np.abs(gy)) #suma de las diferencias. (sirve como filtro de enfoque y nitidez)

    # 5. Decisión final de calidad
    low_variability = (gray_std < MIN_GRAY_STD and sat_std < MIN_SAT_STD) #detecta parches muy homogéneo y con poca variación de color
    low_information = (entropy < MIN_ENTROPY) #detecta parches con baja complejidad visual
    low_structure = (grad_mean < MIN_GRAD_MEAN) #detecta paches muy planos, lisos o borrosos

    if (low_variability and low_information) or (low_information and low_structure): #se descartan parches que cumplen con criterios
        return False

    return True

#Funcion: Normalizacion de etiquetas
def normalize_label(specimen_type):
    specimen_type = str(specimen_type).strip() #nos aseguramos que la etiqueta este en formato texto y eliminamos espacios

    label_map = {                              #Unificamos etiquetas mas simples
        "tumor_tissue": "tumor",
        "normal_tissue": "NAT"
    }

    return label_map.get(specimen_type, None) #Regresa la etiqueta normalizada

#Funcion: Validacion y preparacion de archivo csv
if not CSV_LABELS.exists():
    raise FileNotFoundError(f"No se encontró el CSV original: {CSV_LABELS}")  #se conforma que exista el archivo

df = pd.read_csv(CSV_LABELS)   #se lee el archivo

required_cols = {"Slide_ID", "Case_ID", "Specimen_Type"}  #buscamos las columnas necesarias
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas en el CSV: {missing}") 

df = df[["Slide_ID", "Case_ID", "Specimen_Type"]].copy()  #limpiamos espacios y transformamos a texto

df["Slide_ID"] = df["Slide_ID"].astype(str).str.strip()
df["Case_ID"] = df["Case_ID"].astype(str).str.strip()
df["Specimen_Type"] = df["Specimen_Type"].astype(str).str.strip()

df["label"] = df["Specimen_Type"].apply(normalize_label)  #unificamos las etiquetas
df = df[df["label"].isin(["tumor", "NAT"])].copy()  #nos quedamos solo con mustras tumor y nat

#Procesamiento para slides especificas y correccion de parametros 
#(solo usado para calibrar parametros y arreglar procesamiento en slides especificas)

# SLIDE_TO_FIX = "C3N-01719-22"
# df = df[df["Slide_ID"] == SLIDE_TO_FIX].copy() #analiza solo la slide que seleccionamos

print("Slide a reprocesar:")
print(df[["Slide_ID", "Case_ID", "label"]])

print("Distribución de etiquetas:")
print(df["label"].value_counts(dropna=False))
print("\nTotal de slides únicos a procesar:", df["Slide_ID"].nunique())

#Procesamiento principal de las imagenes WSI

for _, row in df.iterrows():          #recorre cada slide presente en el dataframe
    slide_id = row["Slide_ID"]        #extrae informacion del slide
    patient_id = row["Case_ID"]
    label_name = row["label"]

    svs_path = SVS_DIR / f"{slide_id}.svs"  #construye la ruta del archivo .svs

    if not svs_path.exists():               #verifica que exista la slide
        print(f"[ADVERTENCIA] No existe: {svs_path}")
        continue

    out_dir = PATCH_DIR / label_name / patient_id       #
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nProcesando: {svs_path.name}")

    try:
        slide = openslide.OpenSlide(str(svs_path))  #abrimos la imagen mediante OpenSlide
        full_w, full_h = slide.dimensions           #obtenemos las dimenciones

        # 1. Creacion de la máscara global de tejido
        thumb_rgb = get_thumbnail_rgb(slide, max_size=THUMB_MAX_SIZE)  #genera miniatura de baja resolucion
        tissue_mask = create_tissue_mask(thumb_rgb)                    #crea mascara binaria de tejido

        # Guardar máscara de control
        mask_img = Image.fromarray((tissue_mask * 255).astype(np.uint8)) #cambiamos la mascara binaria a formato visualizable 
        mask_out_dir = MASK_DIR / label_name / patient_id                #creamos las carpetas a partir de la informacion extraida
        mask_out_dir.mkdir(parents=True, exist_ok=True)                  #Guarda el archivo
        mask_img.save(mask_out_dir / f"{patient_id}_{slide_id}_mask_control.png")  #guarda mascara para control visual

        # 2. Apartado de diagnostico de parches
        #(Empleado solo para ajuste experimental de filtros y control de calidad)
        
        # saved_count = 0
        # total_count = 0
        
        # fail_tissue_ratio = 0
        # fail_white = 0
        # fail_saturation = 0
        # fail_dark = 0
        # fail_variability = 0
        # fail_entropy = 0
        # fail_structure = 0

        # for y in range(0, full_h - PATCH_SIZE + 1, PATCH_SIZE):
        #     for x in range(0, full_w - PATCH_SIZE + 1, PATCH_SIZE):
        #         total_count += 1
        
        #         tissue_ratio = get_patch_tissue_ratio(
        #             tissue_mask, x, y, PATCH_SIZE, full_w, full_h
        #         )
        
        #         if tissue_ratio < MIN_TISSUE_RATIO:
        #             fail_tissue_ratio += 1
        #             continue
        
        #         patch = slide.read_region(
        #             (x, y), READ_LEVEL, (PATCH_SIZE, PATCH_SIZE)
        #         ).convert("RGB")
        
        #         patch_rgb = np.array(patch)
        #         patch_float = patch_rgb.astype(np.float32) / 255.0
        
        #         gray = np.mean(patch_rgb, axis=2)
        #         hsv = rgb2hsv(patch_float)
        #         s = hsv[:, :, 1]
        
        #         white_ratio = np.mean(gray > WHITE_PIXEL_THR)
        #         mean_saturation = np.mean(s)
        #         dark_ratio = np.mean(gray < 30)
        
        #         if white_ratio > MAX_WHITE_RATIO:
        #             fail_white += 1
        #             continue
        
        #         if mean_saturation < MIN_MEAN_SATURATION:
        #             fail_saturation += 1
        #             continue
        
        #         if dark_ratio > MAX_DARK_RATIO:
        #             fail_dark += 1
        #             continue
        
        #         gray_std = np.std(gray)
        #         sat_std = np.std(s)
        
        #         hist, _ = np.histogram(gray, bins=32, range=(0, 255), density=True)
        #         hist = hist[hist > 0]
        #         entropy = -np.sum(hist * np.log2(hist))
        
        #         gx = np.diff(gray, axis=1)
        #         gy = np.diff(gray, axis=0)
        #         grad_mean = np.mean(np.abs(gx)) + np.mean(np.abs(gy))
        
        #         low_variability = (gray_std < MIN_GRAY_STD and sat_std < MIN_SAT_STD)
        #         low_information = (entropy < MIN_ENTROPY)
        #         low_structure = (grad_mean < MIN_GRAD_MEAN)
        
        #         if low_variability and low_information:
        #             fail_variability += 1
        #             continue
        
        #         if low_information and low_structure:
        #             fail_structure += 1
        #             continue
        
        #         patch_name = f"{patient_id}_{slide_id}_x{x}_y{y}.png"
        #         patch.save(out_dir / patch_name)
        #         saved_count += 1
        
        # print(f"  Parches evaluados: {total_count}")
        # print(f"  Fallan por tissue_ratio: {fail_tissue_ratio}")
        # print(f"  Fallan por blanco: {fail_white}")
        # print(f"  Fallan por saturación: {fail_saturation}")
        # print(f"  Fallan por oscuro: {fail_dark}")
        # print(f"  Fallan por variabilidad+entropía: {fail_variability}")
        # print(f"  Fallan por entropía+estructura: {fail_structure}")
        # print(f"  Parches guardados: {saved_count}")

        #Extraccion final de parches validos

        saved_count = 0   #contadores
        total_count = 0

      
        for y in range(0, full_h - PATCH_SIZE + 1, PATCH_SIZE):        #bucle para recorre toda la imagen WSI utilizando el PATCH_SIZE (256*256)
            for x in range(0, full_w - PATCH_SIZE + 1, PATCH_SIZE):
                total_count += 1

                tissue_ratio = get_patch_tissue_ratio(                #verificacion de porcentaje de tejido
                    tissue_mask, x, y, PATCH_SIZE, full_w, full_h
                )

                if tissue_ratio < MIN_TISSUE_RATIO:                   #descarte de regioens con poco tejido
                    continue

                patch = slide.read_region(                            #Extrae el parche de la WSI
                    (x, y), READ_LEVEL, (PATCH_SIZE, PATCH_SIZE)
                ).convert("RGB")

                if not patch_quality_filters(patch):                  #aplicamos filtros de calidad adicionales
                    continue

                patch_name = f"{patient_id}_{slide_id}_x{x}_y{y}.png"   #guarda parches validos
                patch.save(out_dir / patch_name)
                saved_count += 1

        slide.close()    #Cierra la slide

        print(f"  Parches evaluados: {total_count}")           #muestra el resumen del slide
        print(f"  Parches guardados: {saved_count}")
        print(f"  Carpeta salida: {out_dir}")

    except Exception as e:                                      #Muestra informacion si falla alguna slide
        print(f"[ERROR] Falló {svs_path.name}: {e}")

print("\nExtracción de parches limpios finalizada.")

Control de numero de parches posterior a la extraccion

In [ ]:
import pandas as pd
from pathlib import Path
import re

#Ruta de los parches extraidos

PATCH_DIR = Path(r"D:\ProyectoPDAC\Parches_PDAC_python")

#Parches minimos permitidos

MIN_PATCHES_ALERTA = 50

#Funcion: Extrae slide especifico usando slide_id o case_id

def extraer_slide_id(nombre_archivo):
    stem = Path(nombre_archivo).stem                                   #obtiene el nombre del archivo sin la extencion
    m = re.match(r"^(C3[NL]-\d+)_(C3[NL]-\d+-\d+)_x\d+_y\d+$", stem)   #Busca el patron esperado en el nombre del parche
    if m:                                                              #si el nombre cumple con el patron procede la extraccion de IDs
        case_id = m.group(1)
        slide_id = m.group(2)
        return case_id, slide_id
    else:                                                              #si no, lo ignora
        return None, None

#Recorre todos los parches

rows = []

for img_path in PATCH_DIR.rglob("*.png"):                             #busca todos los archivos en .png dentro de la carpeta de parches
    if "mask" in img_path.name.lower():                               #ignora los archivos de mascaras
        continue

    rel_parts = img_path.relative_to(PATCH_DIR).parts

    if len(rel_parts) < 3:
        continue

    clase = rel_parts[0]                                              #extrae informacion de clase, carpeta del paciente y nombre del archivo
    carpeta_paciente = rel_parts[1]
    nombre_archivo = rel_parts[-1]

    case_id, slide_id = extraer_slide_id(nombre_archivo)              #extrae case_id y slide_id desde el nombre del parche

    if case_id is None or slide_id is None:                           #ignora nombres que no cumplen con el formato
        continue

    rows.append({                                                     #se guarda la informacion del parche en una lista
        "class": clase,
        "patient_folder": carpeta_paciente,
        "Case_ID": case_id,
        "Slide_ID": slide_id,
        "patch_name": nombre_archivo
    })

#Creamos dataframe a partir de la lista creada

df = pd.DataFrame(rows)

if df.empty:
    print("No se encontraron parches válidos.")
else:
    resumen_slide = (                                  #resumen de parches por slide
        df.groupby(["class", "Case_ID", "Slide_ID"])
          .size()
          .reset_index(name="n_patches")
          .sort_values(["n_patches", "class", "Case_ID", "Slide_ID"], ascending=[True, True, True, True])
          .reset_index(drop=True)
    )

    # identificacion de slides con numero bajo de parches
    resumen_slide["alerta_bajo_minimo"] = resumen_slide["n_patches"] < MIN_PATCHES_ALERTA

    # Guarda el reporte
    out_csv = PATCH_DIR / "reporte_patches_por_slide.csv"
    resumen_slide.to_csv(out_csv, index=False, encoding="utf-8-sig")

    # Mustra el resumen de los 30 slides con menos parches
    print("Reporte generado:", out_csv)
    print("Total de slides:", len(resumen_slide))
    print(f"Slides con menos de {MIN_PATCHES_ALERTA} parches:", (resumen_slide["n_patches"] < MIN_PATCHES_ALERTA).sum())
    print("\nSlides con menos parches:")
    display(resumen_slide.head(30))

    # Guarda el reporte
    pocos = resumen_slide[resumen_slide["n_patches"] < MIN_PATCHES_ALERTA].copy()
    out_csv_pocos = PATCH_DIR / f"reporte_slides_menos_de_{MIN_PATCHES_ALERTA}_parches.csv"
    pocos.to_csv(out_csv_pocos, index=False, encoding="utf-8-sig")

    print(f"\nArchivo de slides problemáticos: {out_csv_pocos}")    

Normalizacion de Color con Macenko para los parches limpios

In [1]:
#Normalizacion de tincion basado en el metodo de Macenko et al. (2009)

import numpy as np           #librerias necesarias
import cv2
from pathlib import Path
import csv
import shutil

#Rutas
PATCH_DIR = Path(r"D:\ProyectoPDAC\Parches_PDAC_python")                                    #parches originales
OUT_DIR = Path(r"D:\ProyectoPDAC\Parches_PDAC_Macenko")                                     #parches normalizados
REFERENCE_PATCH = Path(r"D:\ProyectoPDAC\Referencia_Macenko_Candidatos\imgreferencia.png")  #imagen de referencia
OUT_DIR.mkdir(parents=True, exist_ok=True)                                                  #crea la carpeta de salida si no existe

#Definimos los parametros Macenko (valores basados en Macenko et al. (2009))

Io = 240        #intensidad de luz empleada para calcular la densidad optica
alpha = 1       #percentil robusto para estimar vectores de tincion
beta = 0.10     #Umbral para eliminar pixeles con baja densidad optica

#funciones 

def read_rgb_image(img_path):
    img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)        #se lee la imagen con opencv en formato BGR
    if img_bgr is None:
        raise ValueError("Error al leer la imagen con OpenCV.")  
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)              #convierte de BGR a RGB

def save_rgb_image(img_rgb, out_path):                           
    out_path = Path(out_path)                                    #creamos carpeta 
    out_path.parent.mkdir(parents=True, exist_ok=True)

    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)           #ya que opencv trabaja con formato BGR se transforma nuevamente

    # Guardado normal
    ok = cv2.imwrite(str(out_path), img_bgr)
    if ok:
        return

    # Si falla guardado robusto alternativo
    ext = out_path.suffix.lower()
    success, buffer = cv2.imencode(ext, img_bgr)
    if success:
        buffer.tofile(str(out_path))
        return

    raise ValueError("Error al guardar la imagen con OpenCV.")

def standardize_brightness(img):
    img = img.astype(np.float32)   #tranformamos a decimal (float)
    p = np.percentile(img, 90)     #utilizamos percentil 90 como valor de intencidad luminica para evitar ruido

    if p <= 0:
        return img.astype(np.uint8)   #Evitamos divisiones por 0. regresa la imagen original tal como estaba

    img = img * 255.0 / p    #reajuste de escala de todos los pixeles. toma en cuenta el percentil 90 y lo fuerza como el blanco puro. Los demas pixeles se adaptan proporcionalmente.
    img = np.clip(img, 0, 255)  #recorte de valores extremos debido a la multiplicacion en el paso anterior
    return img.astype(np.uint8) #regresa la imagen al formato de 8 bits. sin decimales

def rgb_to_od(img):
    img = img.astype(np.float32)  #conversion de RGB a densidad optica
    img[img <= 0] = 1  #evitamos ceros antes del log
    return -np.log((img + 1) / Io)  #aplicamos la formula de Beer-Lambert para obtener la densidad optica

def od_to_rgb(od):
    img = (Io * np.exp(-od)) - 1  #conversion inversa de densidad optica a RGB
    img = np.clip(img, 0, 255)  #recorte de valores extremos
    return img.astype(np.uint8) #regresa la imagen al formato de 8 bits

def get_stain_matrix(img, beta=0.10, alpha=1):   #(implemetacion del metodo Macenko) Firma de color
    OD = rgb_to_od(img).reshape((-1, 3))  # Convierte la imagen a espacio de densidad optica
    OD = OD[~np.any(OD < beta, axis=1)]   # Elimina pixeles con baja densidad optica. menor a beta (0.10)

    if OD.shape[0] < 10:
        raise ValueError("Muy pocos píxeles válidos para estimar tinción.")

    cov = np.cov(OD.T)  #matriz de covarianza en espacio de densidad optica
    _, eigvecs = np.linalg.eigh(cov)  #selecionamos dos vectores con mayor varianza
    eigvecs = eigvecs[:, [2, 1]]  #especificamente los componentes principales 1 y 2. simialr a un PCA

    if eigvecs[0, 0] < 0:  #ajuste de orientacion de los vectores, para que sean positivos
        eigvecs[:, 0] *= -1
    if eigvecs[0, 1] < 0:
        eigvecs[:, 1] *= -1

    That = np.dot(OD, eigvecs)  #proyectamos los vectores sobre un plano
    phi = np.arctan2(That[:, 1], That[:, 0])  #calculamos los angulos del plano

    minPhi = np.percentile(phi, alpha)  #seleccionamos los extremos a partir de alpha
    maxPhi = np.percentile(phi, 100 - alpha)

    v1 = np.dot(eigvecs, np.array([np.cos(minPhi), np.sin(minPhi)]))  #reconstruccion de los angulos extremos al spacio de densidad optica
    v2 = np.dot(eigvecs, np.array([np.cos(maxPhi), np.sin(maxPhi)]))

    if v1[0] > v2[0]:  #Se ordena los vectores para formar la matriz HyE. al compara con el canal rojo la hematoxilina siempre quede en la primera columna
        HE = np.array([v1, v2]).T
    else:
        HE = np.array([v2, v1]).T

    return HE

def get_concentrations(img, stain_matrix):  #proceso de deconvolucion de color
    OD = rgb_to_od(img).reshape((-1, 3))  #transformamos el parche de RGB a densidad optica
    C, _, _, _ = np.linalg.lstsq(stain_matrix, OD.T, rcond=None) #usamos minimos cuadrados lienales para calcular la ecaucion matricial
    return C  # la matriz C resultante contiene dos filas. relacionadas con las concentraciones cuantitativas de H y E

def macenko_normalize(img, stain_matrix_target, maxC_target):  #se normaliza el aprche a partir de una imagen de referencia. tomada de la base de datos.
    img_std = standardize_brightness(img)  #se estandariza el brillo del parche original

    stain_matrix_source = get_stain_matrix(img_std, beta=beta, alpha=alpha)  #se estima la matriz de tincion del parche original
    C_source = get_concentrations(img_std, stain_matrix_source)              #se estima concentraciones de tincion del parche original

    maxC_source = np.percentile(C_source, 99, axis=1)  #se calcula la saturacion maxima robusta
    maxC_source[maxC_source == 0] = 1   #evita division por 0

    C_source = C_source / maxC_source[:, None]   #se normaliza las concentraciones del parche original
    C_source = C_source * maxC_target[:, None]   #Ajusta la concentracion del parche ariginal a la referenica

    OD_norm = np.dot(stain_matrix_target, C_source).T #reconstruye la imagen normalizada en el espacio de densidad optica
    img_norm = od_to_rgb(OD_norm.reshape(img.shape))  #convierte la imagen normalizada en densidad optica a RGB

    return img_norm

#carga de imagen de referencia

if not REFERENCE_PATCH.exists():
    raise FileNotFoundError(f"No existe la referencia: {REFERENCE_PATCH}")

target = read_rgb_image(REFERENCE_PATCH)  #leer y estandarizar la imagen de referencia
target = standardize_brightness(target)

stain_matrix_target = get_stain_matrix(target, beta=beta, alpha=alpha) #calcula la matriz de tincion de referencia
C_target = get_concentrations(target, stain_matrix_target)  #se calcula las concentraciones de tincion de referencia
maxC_target = np.percentile(C_target, 99, axis=1)  #obtenemos la concentracion maxima robusta de referencia
maxC_target[maxC_target == 0] = 1

print("Referencia cargada correctamente.")

#Normalizacion de parches

total_normalizados = 0  #contador de parches normalizados
total_fallidos = 0      #contador de parches fallidos

for img_path in PATCH_DIR.rglob("*.png"):

    if "mask" in img_path.name.lower(): #evita procesar archivos auxiliares (mascaras)
        continue

    rel_path = img_path.relative_to(PATCH_DIR)  #mantiene la misma estructura de carpetas en la salida
    out_path = OUT_DIR / rel_path  #define la ruta final donde se guardaran los parches normalizados

    try:
        img = read_rgb_image(img_path)  #lee el parche original

        img_norm = macenko_normalize(    #se aplica la normalización Macenko
            img,
            stain_matrix_target,
            maxC_target
        )

        save_rgb_image(img_norm, out_path)  #guarda parches normalizados

        total_normalizados += 1  #aumenta el contador

        if total_normalizados % 1000 == 0: #Monitor de progreso
            print(f"Parches normalizados: {total_normalizados}")

    except Exception as e:
        total_fallidos += 1   #aumenta el contador
        print(f"[ERROR] No se pudo normalizar {img_path.name}: {e}")

#Resumen de normalizacion

print("\nNormalización Macenko finalizada.")
print("Parches normalizados:", total_normalizados)
print("Parches fallidos:", total_fallidos)
print("Carpeta de salida:", OUT_DIR)


Split por paciente en proporcion 70-15-15

In [ ]:
#Division del dataset en entrenamineto(train), validacion (Val) y prueba (Test)

import pandas as pd    #libreria para trabjar con dataframe. lectura y manejo de tablas
from sklearn.model_selection import train_test_split   #libreria para machine learning

#Carga del indice que contiene la informacion de cada slide. (case_id, slide_id, rutas, etc)

df = pd.read_csv(r"D:\ProyectoPDAC\mil_index_slides.csv")

#Agrupamiento de pacientes segun muestra medica

patient_labels = (                     #cada paciente pude tener solo tumot, NAT o ambos tipos
    df.groupby("Case_ID")["label"]     #Agrupa las filas que pertecencen a un mismo paciente
      .apply(lambda x: sorted(set(x))) #elimina duplicados y ordena alfabericamente
      .reset_index()
)

def assign_patient_profile(labels):  #funcion para clasificacion (3 perfiles)
    labels = set(labels)
    if labels == {"tumor"}:
        return "tumor_only"          #solo tumor
    elif labels == {"NAT"}:
        return "nat_only"            #solo nat
    elif labels == {"tumor", "NAT"}:
        return "mixed_nat_tumor"     #ambas
    else:
        return "other"

patient_labels["patient_profile"] = patient_labels["label"].apply(assign_patient_profile)  #aplicamos la funcion anterior con la lista de pacientes

print("Distribución de perfiles por paciente:")
print(patient_labels["patient_profile"].value_counts()) #muestra el resumen final

#Split 70-15-15 por paciente y estratificado por perfil

train_patients, temp_patients = train_test_split( #dividimos los datos en dos grupos
    patient_labels,   #tabla de pacientes
    test_size=0.30,   #tamaño del split
    random_state=42,  #semilla de control para replicar
    stratify=patient_labels["patient_profile"]  #asegura que halla proporcionalidad de grupos dentro de los pacientes (tumor, NAT y ambos)
)

val_patients, test_patients = train_test_split( #subdividimos nuevamente en dos grupos
    temp_patients,  #usamos solo el 30% de los pacientes
    test_size=0.50, #a este grupo lo dividimos en dos
    random_state=42,
    stratify=temp_patients["patient_profile"]  #otravez estratificamos
)

train_ids = set(train_patients["Case_ID"])  #Creamos llaves de busqueda para cada uno de los splits
val_ids = set(val_patients["Case_ID"])
test_ids = set(test_patients["Case_ID"])

#asignacion de subset a cada slide

def assign_subset(case_id):
    if case_id in train_ids:           #uitilizamos condicionales para evaluar cada paciente (train, val, test)
        return "train"
    elif case_id in val_ids:
        return "val"
    elif case_id in test_ids:
        return "test"
    else:
        return "unknown"

df["subset"] = df["Case_ID"].apply(assign_subset)   #se asigna train, val, test a cada paciente

#Validacion del split

if (df["subset"] == "unknown").any():                      #validamos que todos los pacientes tengan una asignacion
    raise ValueError("Hay pacientes sin subset asignado.")

#Guarda el indice actualizado (agregamos subset)

out_path = r"D:\ProyectoPDAC\mil_index_split.csv"   #ruta de salida
df.to_csv(out_path, index=False)                    #se guarda  el dataframe incluyendo la nueva columna

print("\nSplit guardado en:", out_path)            #muestra donde se guardo el archivo de salida

print("\nDistribución por subset (slides):")       #muestra cuantas slides se asignaron a cada grupo
print(df["subset"].value_counts())

print("\nDistribución por label y subset (slides):") #creamos una tabla de contingencia. revisamos cauntas mustras hay en cada subset
print(pd.crosstab(df["subset"], df["label"]))

#Reporte de pacientes

patient_split_df = patient_labels.copy()      #usamos una copia                                     
patient_split_df["subset"] = patient_split_df["Case_ID"].apply(assign_subset)      #aberiguamos en que subset esta cada paciente

print("\nDistribución por subset (pacientes):")  #cuenta y muestra el total de pacientes asignados a cada conjunto de datos
print(patient_split_df["subset"].value_counts())

print("\nDistribución por perfil y subset (pacientes):")    #tabla cruzada para comprobar que las proporciones de grupos se mantenga iguales en train, val y test
print(pd.crosstab(patient_split_df["subset"], patient_split_df["patient_profile"]))

Extraccion de caracteristicas con UNI

In [ ]:
#Verificamos la GPU 

import torch  #importa libreria PyTorch. para construccion y entrenamiento de redes neuronales
print(torch.cuda.is_available())   #Comprobacion de GPU
print(torch.cuda.get_device_name(0)) #Mustra la primera grafica detectada

In [ ]:
#extraccion de caracteristicas

import torch                                 #importamos librerias necesarias. tensores redes neuronales
import torchvision.transforms as transforms  #transformaciones y aumento de datos
from pathlib import Path    #gestion de rutas de archivos y carpetas
import numpy as np   #libreria matematica
import cv2    #importamos OpenCV para imagenes
import pandas as pd  #para la carga, filtrado y manipulación de las tablas de datos
import timm   #arquitecturas de redes neuronales (UNI)
from huggingface_hub import login           #para autenticacion y descarga de modelos de la plataforma Hugging Face
import re   #motor de extresiones regulares (buscar, limpiar y filtrar)

#Rutas y configuracion

CSV_PATH = r"D:\ProyectoPDAC\mil_index_split.csv"
OUT_DIR = Path(r"D:\ProyectoPDAC\Embeddings_UNI")
OUT_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 32   #numero de imagenes que procesa la grafica al mismo tiempo
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   #usa la grafica, si no esta disponible el cpu

print("Device:", DEVICE)  #muestra el hardware seleccionado para el procesamiento

#ingreso a Hugging Face

login()

# Modelo (UNI)

model = timm.create_model(       #descarga el modelo UNI
    "hf-hub:MahmoodLab/UNI",     #repositorio
    pretrained=True,             #modelo preentrenado con millones de imagenes medicas
    init_values=1e-5,            #valores iniciales de escala para capas de atencion
    num_classes=0,              #Elimina la capa final de clasificación para usar el modelo solo como extractor de características
    dynamic_img_size=True        #el modelo permite procesar imagenes de tamaños variables
)
model = model.to(DEVICE)     #Carga el modelo a la GPU
model.eval()                 #modo de evaluacion (para que no modifique valores internos)

#Transformaciones de imagen

transform = transforms.Compose([
    transforms.ToPILImage(),             #convierte imagen NumPy a formato PIL
    transforms.Resize(224),              #ajusta el tamaño del aprche a 224*224 (parches para formato UNI)
    transforms.ToTensor(),               #transformamos la imagen a tensor
    transforms.Normalize(               #Normalizamos al estandar de ImageNet
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

#funciones

coord_pattern = re.compile(r"_x(\d+)_y(\d+)\.png$", re.IGNORECASE)  #patron para extraer coordenadas x,y directamente del nombre del parche

def read_image(path):
    img = cv2.imread(str(path))  #lee la imagen
    if img is None:
        raise ValueError(f"Error leyendo imagen: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def extract_xy(filename):  
    m = coord_pattern.search(filename)  #busca las coordenadas x,y en el nombre del parche
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2)) #devuelve las coordenadas como numeros enteros

#Indice de slides

df = pd.read_csv(CSV_PATH) #carga el indice

#Procesamiento

for idx, row in df.iterrows():          #Bucle para recorrer fila por fila la tabla y estraer informacion
    slide_id = row["Slide_ID"]           #id de la slide
    patch_dir = Path(row["patch_dir"])  #ruta de la carpeta donde esta guardada

    out_path = OUT_DIR / f"{slide_id}.npz" #ruta para guardar archivo de embeddings de salida

    if out_path.exists():             #si ya esxiste el archivo evita reprocesar el slide
        print(f"[SKIP] {slide_id}")
        continue

    patch_files = sorted(patch_dir.glob(f"*_{slide_id}_x*_y*.png")) #busca unicamente solo parches de este slide

    if len(patch_files) == 0:
        print(f"[WARN] Sin parches para {slide_id}")  #alerta si la carpeta esta vacia
        continue
                                 #creamos listas en blanco para guardar los resultados
    embeddings = []              #vectores numericos (caracteristicas) generadas por UNI
    patch_names = []             #nombres de archivo de cada parche procesado
    xs = []                      #coordenadas horizontales
    ys = []                      #coordenadas verticales
    batch = []                   #acumulador temporal de tensores de imagenes para enviar a la GPU
    batch_meta = []              #acumulador temporal de metadatos (nombres) para el lote actual (batch)

    print(f"\nProcesando slide: {slide_id} ({len(patch_files)} parches)")

    for p in patch_files:    #recorre los parches uno por uno de cada slide
        try:
            img = read_image(p) #lectura de la imagen
            img = transform(img) #aplicamos transformacion requerida para UNI
            batch.append(img)  #agrega imagen al batch
            batch_meta.append(p.name) #guarda el nombre del parche

            if len(batch) == BATCH_SIZE:  #si el lote actual alcanza el maximo de 32 se envia a la grafica (GPU)
                batch_tensor = torch.stack(batch).to(DEVICE)

                with torch.inference_mode(): #desactivamos calculo de gradientes. para liberar memoria.
                    feats = model(batch_tensor)

                feats_np = feats.cpu().numpy().astype(np.float32)  #extrae los embedding de lote usando el modelo UNI

                for feat, name in zip(feats_np, batch_meta):   #desempaqueta los resultados para procesar parche por parche
                    x, y = extract_xy(name)
                    embeddings.append(feat)
                    patch_names.append(name)  #extrae numericamente las coordenadas x e y a partir del nombre
                    xs.append(x)
                    ys.append(y)

                batch = []               #vacia los listas para de los lotes para iniciar una nueva recoleccion de nuevos parches
                batch_meta = []

        except Exception as e:
            print(f"Error en {p.name}: {e}")   #evitamos que se detenga todo el proceso si hay un error y lo mustra
            continue

    # Procesamiento para el ultimo batch si es que hay menos de 32
    if len(batch) > 0:
        batch_tensor = torch.stack(batch).to(DEVICE)

        with torch.inference_mode():
            feats = model(batch_tensor)

        feats_np = feats.cpu().numpy().astype(np.float32)

        for feat, name in zip(feats_np, batch_meta):
            x, y = extract_xy(name)
            embeddings.append(feat)
            patch_names.append(name)
            xs.append(x)
            ys.append(y)

    embeddings = np.array(embeddings, dtype=np.float32)   #convierte los resultados en arreglo NumPy
    xs = np.array(xs, dtype=np.int32)
    ys = np.array(ys, dtype=np.int32)
    patch_names = np.array(patch_names)

    np.savez_compressed(    #guardar embeddings y metadatos del slide
        out_path,
        embeddings=embeddings,
        x=xs,
        y=ys,
        patch_names=patch_names
    )

    print(f"Guardado: {slide_id} → {embeddings.shape}") #muestra el nombre guardado y las dimensiones finales de la matriz

print("\nEmbeddings UNI completos.") #mensaje de finalizacion

Comprobacion de embeddings UNI y splits

In [2]:
#importamos librerias

from pathlib import Path
import pandas as pd
import numpy as np

#cargamos rutas

CSV_PATH = Path(r"D:\ProyectoPDAC\mil_index_split.csv") #indice
EMB_DIR = Path(r"D:\ProyectoPDAC\Embeddings_UNI")   #embenddings

In [ ]:
df = pd.read_csv(CSV_PATH) #lee el archivo csv y lo convierte en dataframe

label_map = {"NAT": 0, "tumor": 1}  #definimos un diccionario para transformar las etiquetas
df["y"] = df["label"].map(label_map) #creamos una columna y aplicamos la transformacion

df["embedding_path"] = df["Slide_ID"].apply(lambda x: str(EMB_DIR / f"{x}.npz"))  #creamos la ruta del archivo en fromato .npz a cada slide
df = df[df["embedding_path"].apply(lambda x: Path(x).exists())].copy()  #conserva solo slides que tienen embeddings 

print("Columnas actuales:", df.columns.tolist()) #mustra la lista de pacientes con la ruta de embedding creados

In [ ]:
print("Slides con embeddings:", len(df)) #mustra el total de slides que tienen archivo con embeddings
print(df["subset"].value_counts()) #mustra cuantas slides en cada subconjunto
print(df["label"].value_counts()) #muestra cuantas muestras son de NAT y cuantas de Tumor en el dataset total

train_df = df[df["subset"] == "train"].copy()  #filtra y crea una copia del dataframe para entrenamiento, validacioon y evaluacion
val_df   = df[df["subset"] == "val"].copy()
test_df  = df[df["subset"] == "test"].copy()

print("Train:", len(train_df))   #mustra el numero de slides validadas que se usaran para cada split
print("Val:", len(val_df))
print("Test:", len(test_df))

Modelo MIL - Attention

In [ ]:
#librerias

import os #interaccion con el sistema operativo
import numpy as np  #para operaciones numericas
import pandas as pd  #manejo de tablas
from pathlib import Path #manejo de rutas

import torch #tensores y GPU
import torch.nn as nn #capas neuronales
from torch.utils.data import Dataset, DataLoader #para crear dataset personalizados y cargarlos por lotes
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report #metricas de evaluacion

#Rutas 
CSV_PATH = Path(r"D:\ProyectoPDAC\mil_index_split.csv") #indice
EMB_DIR = Path(r"D:\ProyectoPDAC\Embeddings_UNI") #embeddingd
MODEL_OUT = Path(r"D:\ProyectoPDAC\mil_attention_uni_best.pt") #modelo, archivo de salida

#Configuracion
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" #prioriza el GPU
SEED = 42 #semilla 
EPOCHS = 30 #numero de epocas
LR = 1e-4 #tasa de aprendisaje inicial
WEIGHT_DECAY = 1e-5 #factor de penalizacion
INPUT_DIM = 1024   # dimensiones

torch.manual_seed(SEED) #semilla aletoria para operaciones PyTorch
np.random.seed(SEED) #semilla aletoria para operaciones NumPy

print("Device:", DEVICE)

#Carga de indice

df = pd.read_csv(CSV_PATH)

#etiquetas
label_map = {"NAT": 0, "tumor": 1}  #diccionario binario
df["y"] = df["label"].map(label_map) #creacion de la columna para la nueva etiqueta binaria

#Embeddings
df["embedding_path"] = df["Slide_ID"].apply(lambda x: str(EMB_DIR / f"{x}.npz"))  #Asocia cada slide con su archivo en formato .npz
df = df[df["embedding_path"].apply(lambda x: Path(x).exists())].copy()            #Se verifica que exista el archivo

print("Slides con embeddings:", len(df)) #muestra el total de embeddings
print(df["subset"].value_counts()) #mustra la cantidad asignada a cada subset (train, val, test)
print(df["label"].value_counts()) #Mustra el numero de slides calsificadas como Nat o Tumor

train_df = df[df["subset"] == "train"].copy()  #filtra y crea una copia del dataframe para entrenamiento, validacioon y evaluacion
val_df   = df[df["subset"] == "val"].copy()
test_df  = df[df["subset"] == "test"].copy()

#Dataset personalizado para MIL

class MILDataset(Dataset):     #creamos una clase personalizada de PyTorch
    def __init__(self, dataframe):  #se ejecuta al crear el dataset (estructura especial de PyTorch)
        self.df = dataframe.reset_index(drop=True)  #guarda el dataframe y reinicia los indices

    def __len__(self):
        return len(self.df)  #regresa el numero total de slides del dataset

    def __getitem__(self, idx):  #carga un slide especifico usando el indice
        row = self.df.iloc[idx]  #obtiene la fila correspondiente al slide

        data = np.load(row["embedding_path"], allow_pickle=True) #carga de embeddingd del slide
        x = data["embeddings"]   # matris de embeddings del slide (instancias(numero de parches), 1024)
        y = np.array(row["y"], dtype=np.float32) #cargamos la etiqueta binaria (Nat=0, Tumor=1)

        x = torch.tensor(x, dtype=torch.float32) #transformamos a tensores
        y = torch.tensor(y, dtype=torch.float32)

        meta = {                          #Diccionario para metadatos
            "slide_id": row["Slide_ID"],
            "case_id": row["Case_ID"],
            "label_str": row["label"]
        }

        return x, y, meta  #regresa los tensores y la metadata asociada

#Funcion de agregacion para procesar lotes con tamaño dinamico

def mil_collate(batch):
    xs, ys, metas = zip(*batch) #sepoara embeddings, etiquetas y metadatos
    return xs[0], ys[0], metas[0] #regresa solo el slide actual

train_ds = MILDataset(train_df)  #se crean los tres datasets 
val_ds   = MILDataset(val_df)
test_ds  = MILDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=mil_collate) #se cran tres dataloader (entregan los slides automaticamente)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=mil_collate)
test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=mil_collate)

#Modelo Multiple Instance Learning (MIL) con atenccion

class AttentionMIL(nn.Module): #Se define la red neuronal basado en MIL
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=256, attn_dim=128, dropout=0.25): #se define capas de la red y dimenciones de los espacios ocultas
        super().__init__() #activamos las herramientas internas de PyTorch

        self.feature_proj = nn.Sequential( #reduccion de embeddings
            nn.Linear(input_dim, hidden_dim), #Comprimimos el vector de 1048 a 256
            nn.ReLU(), #activacion no lineal (patrones complejos)
            nn.Dropout(dropout) #Apaga neuronas al azar para evitar el sobreajuste
        )

        self.attention = nn.Sequential(  #aprende que parches son mas importantes para que el modelo clasifique
            nn.Linear(hidden_dim, attn_dim), #proyecta las caracteristicas ocultas al espacio de atenccion
            nn.Tanh(), #activacion no lineal
            nn.Linear(attn_dim, 1) #score de atencion por parche
        )

        self.classifier = nn.Linear(hidden_dim, 1) #genera la prediccion

    def forward(self, x):   #definimos el flujo de datos a partir de la matriz x = (N parches, 1024 caracteristicas)
        
        h = self.feature_proj(x)          #proyecta las 1024 caracteristicas al tamaño oculto
        a = self.attention(h)             #calcula el porcentaje o score de importancia para cada N
        a = torch.softmax(a, dim=0)       #convierte los puntajes brutos en probabilidades que suman 1 en total
        m = torch.sum(a * h, dim=0)       #multiplica cada parche por su peso de atencion y los suma a todos
        logit = self.classifier(m)        #el vector pasa por la capa final para obtener un porcentaje de clasificacion
        return logit.squeeze(0), a.squeeze(1) #regresa el porcentaje de clasificacion y los pesos de atenccion

model = AttentionMIL().to(DEVICE) #llamado de la arquitectura a ejecutarce en la GPU

#Tratamiento para desbalance de clases

n_neg = (train_df["y"] == 0).sum() #contador de mustras negativas (clase 0 o NAT)
n_pos = (train_df["y"] == 1).sum() #contador de mustras positivas (clase 1 o Tumor)

pos_weight_value = n_neg / n_pos if n_pos > 0 else 1.0 #calcula el factor de balanceo (negativos/positivos)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE) #convierte el factor a tensor PyTorch

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight) #calcula el error del modelo usando el peso para equilibrar las clases
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY) #configura el optimizador (Adam) que ajsuta los pesos para el aprendisaje

print("pos_weight:", pos_weight_value) #mustra el valor del peso final

#Funciones

def run_epoch(model, loader, criterion=None, optimizer=None, train=False): #definimos la funcion para procesar una epoca de datos
    if train:
        model.train() #se activa el modo entrenamiento
    else:
        model.eval() #se activa el modo evaluacion

    losses = []  #listas para almacenar
    y_true = []
    y_prob = []

    for x, y, meta in loader:  #bucle para recorrer las slides
        x = x.to(DEVICE)   #se envia la matriz de parches de la slide a la GPU
        y = y.to(DEVICE)   #se envia la etiqueta correspondiente a la slide a la GPU

        if train:
            optimizer.zero_grad() #borra los gradientes anteriores

        with torch.set_grad_enabled(train): #se abilita el calculo de gradientes
            logit, attn = model(x) #la slide pasa por la res neuronal para obtener la prediccion y atencion
            loss = criterion(logit.view(1), y.view(1)) #calcula el error de la slide. para corregir el error y aprender

            if train:
                loss.backward() #se calcula como afecto el error a cada peso en la red
                optimizer.step() #reajusta los pesos en la red

        prob = torch.sigmoid(logit).item() #transforma la prediccion en probabilidad (entre 1 y 0)
        #guardamos los resultados en las listas creadas respectivamente
        losses.append(loss.item())   #el error del slide
        y_true.append(int(y.item())) #el diagnostico, la etiqueta entre 0 o 1
        y_prob.append(prob)          #la probabilidad predicha

    y_pred = [1 if p >= 0.5 else 0 for p in y_prob] #se usa las probabilidades para desiciones (1 si es igual o mayor a 0.5, sino 0)
    #calculamos las metricas finales
    metrics = {
        "loss": float(np.mean(losses)) if losses else np.nan,
        "acc": accuracy_score(y_true, y_pred) if y_true else np.nan,
        "f1": f1_score(y_true, y_pred, zero_division=0) if y_true else np.nan,
        "auc": roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else np.nan
    }

    return metrics, y_true, y_prob #devuelve metricas, el diagnostico real y la prediccion

#Entrenamiento

best_val_auc = -np.inf #se usa el mejor AUC obtenido
history = []  #lista para guardar metricas de cada epoca

for epoch in range(1, EPOCHS + 1): #bucle que entrena el numero de epocas predefinido
    train_metrics, _, _ = run_epoch( #ejecuta cada epoca de entrenamiento
        model, train_loader, criterion=criterion, optimizer=optimizer, train=True
    )

    val_metrics, _, _ = run_epoch( #ejecuta cada epoca de validacion
        model, val_loader, criterion=criterion, optimizer=None, train=False
    )

    history.append({            #se guardan metricas de entrenamiento y validacion
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "train_f1": train_metrics["f1"],
        "train_auc": train_metrics["auc"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_f1": val_metrics["f1"],
        "val_auc": val_metrics["auc"],
    })

    print(                     #se muestra el resumen de cada epoca
        f"Epoch {epoch:02d} | "
        f"Train loss {train_metrics['loss']:.4f} acc {train_metrics['acc']:.4f} auc {train_metrics['auc']:.4f} | "
        f"Val loss {val_metrics['loss']:.4f} acc {val_metrics['acc']:.4f} auc {val_metrics['auc']:.4f}"
    )

    current_val_auc = val_metrics["auc"] #AUC actual de validacion
    if not np.isnan(current_val_auc) and current_val_auc > best_val_auc:
        best_val_auc = current_val_auc #actualiza al mejor encontrado
        torch.save(model.state_dict(), MODEL_OUT) #guarda pesos del modelo
        print(f" Mejor modelo guardado en: {MODEL_OUT}") #muestra donde se guarda el mejor modelo


#Evaluacion o test

print("\nMejor modelo")
model.load_state_dict(torch.load(MODEL_OUT, map_location=DEVICE)) #se carga el mejor modelo guardado en validacion

test_metrics, y_true_test, y_prob_test = run_epoch( #se ejecuta utilizando el conjunto test
    model, test_loader, criterion=criterion, optimizer=None, train=False
)

y_pred_test = [1 if p >= 0.5 else 0 for p in y_prob_test] #se usa las probabilidades para desiciones (1 si es igual o mayor a 0.5, sino 0)

print("\nResultados Test")                   #se muestran las metricas de evaluacion o test
print("Loss:", test_metrics["loss"])     #perdida (loss) promedio del conjunto test
print("Accuracy:", test_metrics["acc"]) #proporcion de predicciones correctas
print("F1:", test_metrics["f1"])        #balance entre precision y sensibilidad
print("AUC:", test_metrics["auc"])      #capacidad discriminativa del modelo

print("\nMatriz de confusion:")
print(confusion_matrix(y_true_test, y_pred_test)) #se genera una matriz de confusion simple

print("\nReporte de clasificación:")
print(classification_report(                 #se genera un reporte de metricas detalladas por clase
    y_true_test,
    y_pred_test,
    target_names=["NAT", "tumor"],
    zero_division=0
))

Visualizacion de parches

In [ ]:
#Preparacion de datos

#Librerias
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt #para visualizacion de imagenes y graficas
from pathlib import Path
from PIL import Image  #para lecturas de parches de imagenes

#Rutas
CSV_PATH = Path(r"D:\ProyectoPDAC\mil_index_split.csv")
EMB_DIR = Path(r"D:\ProyectoPDAC\Embeddings_UNI")
MODEL_OUT = Path(r"D:\ProyectoPDAC\mil_attention_uni_best.pt")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu" #seleccion de GPU

#Indice
df = pd.read_csv(CSV_PATH)

#Etiquetas y embeddings
label_map = {"NAT": 0, "tumor": 1} #etiquetas
df["y"] = df["label"].map(label_map)
df["embedding_path"] = df["Slide_ID"].apply(lambda x: str(EMB_DIR / f"{x}.npz"))
df = df[df["embedding_path"].apply(lambda x: Path(x).exists())].copy()

test_df = df[df["subset"] == "test"].copy() #filtramos test

print("Slides test disponibles:", len(test_df)) #mustra slides disponibles
print(test_df[["Slide_ID", "Case_ID", "label", "num_patches"]].head()) #muestra algunos como ejemplo

In [ ]:
#Seleccion del slide a visualizar
SLIDE_TO_VIEW = "C3L-00102-21"# ejemplo para tumor
# SLIDE_TO_VIEW = "C3L-00102-26"#ejemplo para NAT

#busca la fila del slide seleccionado
row = test_df[test_df["Slide_ID"] == SLIDE_TO_VIEW].iloc[0]

#extraes la informacion solicitada
slide_id = row["Slide_ID"]
case_id = row["Case_ID"]
label_str = row["label"]
patch_dir = Path(row["patch_dir"])
embedding_path = Path(row["embedding_path"])

#muestra la informacion solicitada
print("Slide:", slide_id)
print("Paciente:", case_id)
print("Label real:", label_str)
print("Patch dir:", patch_dir)
print("Embedding:", embedding_path)

In [ ]:
#extraccion de probabilidad de prediccion y pesos de atenccion

# cargar embeddings del slide
data = np.load(embedding_path, allow_pickle=True) #carga el archivo .npz del slide seleccionado

#Extraccion de la informacion del archivo
x = data["embeddings"]        #embeddings
patch_names = data["patch_names"] #nombre del parche
xs = data["x"] #coordenada x
ys = data["y"] #coordenada y

x = torch.tensor(x, dtype=torch.float32).to(DEVICE) #tranformacion a tensor

with torch.no_grad(): #desactivamos el calculo de gradientes (el modelo ya esta entrenado, no utiliza)
    logit, attn = model(x) #usa eltensor y devuelve logit(el valor de la ultima capa del modelo) y attn (matriz de pesos de atenccion)
    prob = torch.sigmoid(logit).item() #tranformamos el valor de prediccion (logit) a probabilidad

attn = attn.detach().cpu().numpy() #transformamos las atecciones a arreglso numericos NumPy

pred_label = "tumor" if prob >= 0.5 else "NAT" #clasificacion por umbral de 0.5

#mustra resultados
print("Probabilidad tumor:", round(prob, 4)) #probabilidad
print("Predicción:", pred_label) #prediccion
print("Número de instancias:", len(attn)) #total de parches
print("Suma atención:", attn.sum()) #verificacion

print("Shape embeddings:", x.shape)  #dimensiones del tensor de embeddings
print("Primer patch:", patch_names[0] if len(patch_names) > 0 else "N/A") #Nombre del parche

In [ ]:
#Busqueda de parches del slide seleccionado
patch_files = sorted(patch_dir.glob(f"*_{slide_id}_x*_y*.png"))

print("Parches encontrados:", len(patch_files)) #muestra numero de parches encontrados
print("Atenciones:", len(attn)) #muestra numero de pesos de atencciones encontradas

In [ ]:
#Visualizacion de top parches con mayor atenccion

#Libreria
from pathlib import Path

#Numero de parches con atencion a mostrar
TOP_K = 12

# usa los nombres de patch_names del archivo .npz para mantener el mismo orden (embeddings, nombres de parches y valores de atenccion)
patch_paths = [patch_dir / str(name) for name in patch_names]

#Verificacion
print("Número de patch_paths:", len(patch_paths))
print("Número de atenciones:", len(attn))

#verifica si cada parche tiene una atencion asociada
if len(patch_paths) != len(attn):
    raise ValueError("No coincide el número de parches con el número de atenciones.")

#Tabla con parches de atenccion
attn_df = pd.DataFrame({
    "patch_path": [str(p) for p in patch_paths], #ruta
    "attention": attn #velor de atenccion
})

#Ordena de mayor a menor
attn_df = attn_df.sort_values("attention", ascending=False).reset_index(drop=True)

#Selecciona los mejores 12 parches
top_df = attn_df.head(TOP_K)

#Grafica de 3x4
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
axes = axes.flatten()

#Recorre cada eje de la figura y cada poarche seleccionado
for ax, (_, r) in zip(axes, top_df.iterrows()):
    img = Image.open(r["patch_path"]).convert("RGB") #abre el parche
    ax.imshow(img) #muestra el parche
    ax.set_title(f"A={r['attention']:.4f}", fontsize=10) #muestra el valor de atenccion encima del parche
    ax.axis("off") #no muestra ejes

#apaga ejes vacios si hay menos de 12
for ax in axes[len(top_df):]:
    ax.axis("off")

#Titulo
plt.suptitle(
    f"Top {TOP_K} parches (atención)\n"
    f"Slide: {slide_id} | Real: {label_str} | Pred: {pred_label} | p={prob:.3f}"
)

plt.tight_layout() #Ajuste del margen de la figura

#Guarda
out_path = Path(r"D:\ProyectoPDAC\Resultados\Attention")
out_path.mkdir(parents=True, exist_ok=True) #crea si no hay

file_name = f"attention_top_{slide_id}.png" #nombre de la imagen de salida
full_path = out_path / file_name #ruta de la imagen

plt.savefig(full_path, dpi=300, bbox_inches="tight") #
print("Figura guardada en:", full_path)

plt.show() #muestra